In [1]:
import numpy

In [2]:
from data.loaders import MyoGymLoader, RecoFitLoader
from data.preprocessing import create_windows, setup_binary_classification

In [3]:
def binarize_activity(df, non_lifting_activities):
    """
    Sets activity to 0 if it's in the non_lifting_activities keys, 
    otherwise sets it to 1.
    """
    # Get the list of IDs that count as "non-active"
    non_active_ids = list(non_lifting_activities.keys())
    
    # If the value is in our list, it becomes 0 (False), else 1 (True)
    df["activity"] = (~df["activity"].isin(non_active_ids)).astype(int)
    
    return df

In [4]:
def get_binary_stats(labels, label_name="Dataset"):
    """
    Calculates and prints distribution statistics for binary labels.
    """
    unique, counts = numpy.unique(labels, return_counts=True)
    # Ensure we handle cases where only one class might be present
    count_dict = dict(zip(unique, counts))
    c0 = count_dict.get(0, 0)
    c1 = count_dict.get(1, 0)
    total = len(labels)
    
    stats = {
        "Ratio Non-exercise": c0 / total,
        "Ratio Exercise": c1 / total,
        "Total Samples": total,
        "Counts (0, 1)": (c0, c1)
    }
    
    print(f"--- Stats for {label_name} ---")
    for key, value in stats.items():
        print(f"{key}: {value}")
    print("-" * 30)
    return stats

In [5]:
def process_and_window(loader, data_df, mapping):
    """
    Handles windowing and binary setup in one go.
    """
    x, y, meta = create_windows(data_df)
    y_binary = setup_binary_classification(
        y, 
        mapping, 
        loader.get_non_lifting_activities()
    )
    return x, y_binary, meta

In [6]:
def total_time(df):
    # Total time for the entire dataset
    total_seconds = len(df) / 50
    print(f"Total time: {total_seconds} seconds (or {total_seconds/60/60:.2f} hours)")

    # Time per trainer (since each trainer likely has their own recording session)
    trainer_times = df.groupby("trainer").size() / 50 / 60
    print(trainer_times)

In [7]:
myogym_loader = MyoGymLoader("../data/datasets/MyoGym.mat")
myogym_data_df, myogym_mapping = myogym_loader.load_data()

Rows with overlapping timestamps: 408132


In [8]:
# Binarize and get DF stats
mg_binary_df = binarize_activity(myogym_data_df, myogym_loader.get_non_lifting_activities())
get_binary_stats(mg_binary_df["activity"], "MyoGym Raw DF")

--- Stats for MyoGym Raw DF ---
Ratio Non-exercise: 0.7689593150519765
Ratio Exercise: 0.23104068494802354
Total Samples: 1608998
Counts (0, 1): (np.int64(1237254), np.int64(371744))
------------------------------


{'Ratio Non-exercise': np.float64(0.7689593150519765),
 'Ratio Exercise': np.float64(0.23104068494802354),
 'Total Samples': 1608998,
 'Counts (0, 1)': (np.int64(1237254), np.int64(371744))}

In [9]:
total_time(mg_binary_df)

Total time: 32179.96 seconds (or 8.94 hours)
trainer
1     67.264333
2     54.702667
3     54.555667
4     60.333667
5     47.911667
6     66.004000
7     59.978000
8     37.229000
9     53.560667
10    34.793000
dtype: float64


In [10]:
# Windowing and window stats
mg_x, mg_y_binary, mg_meta = process_and_window(myogym_loader, myogym_data_df, myogym_mapping)
get_binary_stats(mg_y_binary, "MyoGym Windows")

--- Stats for MyoGym Windows ---
Ratio Non-exercise: 0.7693455619462148
Ratio Exercise: 0.23065443805378516
Total Samples: 32165
Counts (0, 1): (np.int64(24746), np.int64(7419))
------------------------------


{'Ratio Non-exercise': np.float64(0.7693455619462148),
 'Ratio Exercise': np.float64(0.23065443805378516),
 'Total Samples': 32165,
 'Counts (0, 1)': (np.int64(24746), np.int64(7419))}

In [11]:
recofit_loader = RecoFitLoader("../data/datasets/RecoFit")
recofit_data_df, recofit_mapping = recofit_loader.load_data()

In [12]:
rf_binary_df = binarize_activity(recofit_data_df, recofit_loader.get_non_lifting_activities())
get_binary_stats(rf_binary_df["activity"], "RecoFit Raw DF")

--- Stats for RecoFit Raw DF ---
Ratio Non-exercise: 0.6774875215144794
Ratio Exercise: 0.32251247848552056
Total Samples: 14295605
Counts (0, 1): (np.int64(9685094), np.int64(4610511))
------------------------------


{'Ratio Non-exercise': np.float64(0.6774875215144794),
 'Ratio Exercise': np.float64(0.32251247848552056),
 'Total Samples': 14295605,
 'Counts (0, 1)': (np.int64(9685094), np.int64(4610511))}

In [13]:
total_time(rf_binary_df)

Total time: 285912.1 seconds (or 79.42 hours)
trainer
0      42.653000
1      30.108667
2      35.771000
3      39.933000
4      44.548000
         ...    
121    16.419667
122    37.681333
123    52.490333
124    43.300667
125    42.639667
Length: 126, dtype: float64


In [14]:
rf_x, rf_y_binary, rf_meta = process_and_window(recofit_loader, recofit_data_df, recofit_mapping)
get_binary_stats(rf_y_binary, "RecoFit Windows")

--- Stats for RecoFit Windows ---
Ratio Non-exercise: 0.6774974801209542
Ratio Exercise: 0.3225025198790458
Total Samples: 285728
Counts (0, 1): (np.int64(193580), np.int64(92148))
------------------------------


{'Ratio Non-exercise': np.float64(0.6774974801209542),
 'Ratio Exercise': np.float64(0.3225025198790458),
 'Total Samples': 285728,
 'Counts (0, 1)': (np.int64(193580), np.int64(92148))}

In [15]:
binary_df = binarize_activity(myogym_data_df, myogym_loader.get_non_lifting_activities())

In [16]:
unique, counts = numpy.unique(binary_df["activity"], return_counts=True)
total = len(binary_df["activity"])
counts[0] / total, counts[1] / total, total, counts

(np.float64(0.7689593150519765),
 np.float64(0.23104068494802354),
 1608998,
 array([1237254,  371744]))

In [17]:
p = 0.886
r = 0.887
(2*p*r)/(p+r)

0.8864997179921037